# DenseAM force-matching on 12×12 MNIST 0/1\n\nThis notebook is the §13.3 continuation of [`latent_dam_2d_toy_jax_v2.ipynb`](latent_dam_2d_toy_jax_v2.ipynb) on the MNIST setup proposed in `notes/latent_dam_force_matching.pdf`. It restricts attention to the **categorical DenseAM** model (the log-sum-exp prototype free energy) — none of the polynomial / binary-RBM / MLP baselines from the 2D notebook are included.\n\n**What is kept from the 2D notebook.** The mathematical setup is unchanged. The forward Ornstein–Uhlenbeck corruption $x_t = a_t x_0 + \\sigma_t \\epsilon$ with $a_t = e^{-t}$ and $\\sigma_t^2 = k_BT(1 - a_t^2)$ defines the denoising target $f^\\star = -(x_t - a_t x_0)/\\sigma_t^2$. The DenseAM free energy is\n$$\\log p_\\theta(x, t) \\;=\\; -\\frac{1}{2\\sigma_t^2}\\lVert x\\rVert^2 \\;+\\; \\log \\sum_{a=1}^{K}\\exp\\!\\left[\\frac{a_t\\, x^\\top c_a}{\\sigma_t^2} - \\frac{a_t^2\\lVert c_a\\rVert^2}{2\\sigma_t^2} + b_a\\right] + \\text{const},$$\nwith trainable prototypes $c_a \\in \\mathbb R^d$ and biases $b_a$. The force is the responsibility-weighted attraction $f_\\theta = (1/\\sigma_t^2)\\sum_a r_a(a_t c_a - x)$ with $r_a = \\mathrm{softmax}_a[\\,a_t x^\\top c_a/\\sigma_t^2 - a_t^2 \\lVert c_a\\rVert^2/(2\\sigma_t^2) + b_a\\,]$. The training protocol is unchanged: denoising force matching, per-slice from high noise to low noise, warm-started from the previous slice (§8 of the note).\n\n**What is different from the 2D version.**\n1. **Visible dimension** $d = 144$ instead of $d = 2$. Prototypes are stored 12×12 images.\n2. **Drift-aware prototypes.** In 2D we used $\\tfrac12\\lVert x - c_a\\rVert^2/\\sigma_t^2$ in the logits, which mixes the noised data variance into the Gaussian quadratic. In high-$d$ the OU drift $a_t = e^{-t}$ is no longer negligible across the schedule, so we use the *exact* identity from eq. (5) of the note — the prototypes are matched to clean memories $\\xi_\\mu$ and the logits use $a_t x^\\top c_a / \\sigma_t^2 - a_t^2 \\lVert c_a\\rVert^2/(2\\sigma_t^2)$. At $a_t = 1$ (the variance-exploding limit) this reduces to the 2D form.\n3. **Data**: 12×12 downsampled MNIST restricted to digits 0/1, as specified in §13.3 of the note.\n4. **$K$-sweep diagnostics**: a small sweep over the number of prototypes $K \\in \\{16, 32, 64, 128\\}$ probes the memorization-vs-generalization phase that §9.2/§12.3 of the note flags as the central risk.\n5. **Memorization diagnostic**: mean nearest-neighbour distance from generated samples to the training set, both in pixel space and to the nearest prototype. The note (§13.3) lists this and the spurious-state rate as the two key DenseAM evaluation metrics.\n\nThe interpretation checklist at the bottom of the 2D notebook (gradient check, sample quality, score-error, hidden-time-scale) applies here too, except that the binary-RBM hidden-time-scale sweep is no longer relevant — DenseAM has only one hidden time scale (the categorical posterior).

In [ ]:
import math\nimport time\nfrom functools import partial\n\nimport numpy as np\nimport jax\nimport jax.numpy as jnp\nfrom jax import random\nfrom jax.scipy.special import logsumexp\nimport matplotlib.pyplot as plt\n\njax.config.update("jax_enable_x64", False)\n\nprint("JAX version:", jax.__version__)\nprint("Devices:", jax.devices())

## Configuration\n\nDefaults are calibrated for `RESEARCH_MODE = False` to run in a few minutes on a single GPU; `RESEARCH_MODE = True` widens every knob for cleaner figures.\n\nThe time grid is denser than the 2D toy because the higher dimensionality means the score-error landscape is more curved in $t$: with too-coarse a grid the warm start jumps too far between slices and the optimizer has to relearn rather than refine.

In [ ]:
# ----------------------------------------------------------------------\n# Experiment knobs.\n# ----------------------------------------------------------------------\nSEED = 0\nRESEARCH_MODE = False\n\n# Dataset: 12x12 downsampled MNIST, restricted to digits in DIGITS.\nIMG_SIDE = 12\nIMG_DIM = IMG_SIDE * IMG_SIDE\nDIGITS = (0, 1)\n\nif RESEARCH_MODE:\n    N_TRAIN = 4000\n    N_TIME = 24\n    STEPS_PER_TIME = 500\n    BATCH_SIZE = 256\n    N_REVERSE_STEPS = 400\n    N_SAMPLES = 64\nelse:\n    N_TRAIN = 2000\n    N_TIME = 16\n    STEPS_PER_TIME = 250\n    BATCH_SIZE = 128\n    N_REVERSE_STEPS = 250\n    N_SAMPLES = 64\n\n# Physical / noising parameters.\nkBT = 1.0\nTAU = 3.0\nT_MIN = 2e-2\nTIME_GRID = jnp.linspace(T_MIN, TAU, N_TIME)\n\n# Default DenseAM size; the K-sweep cell overrides this.\nK_DEFAULT = 64\n\n# Optimisation.\nLR_CAT = 3e-3\nGRAD_CLIP = 5.0\nREG_SCALE = 1e-6\n\nkey = random.PRNGKey(SEED)\nprint("config: TAU={}, N_TIME={}, STEPS_PER_TIME={}, K_DEFAULT={}".format(\n    TAU, N_TIME, STEPS_PER_TIME, K_DEFAULT))

## Minimal PyTree Adam\n\nLifted verbatim from the 2D notebook so the file stays self-contained.

In [ ]:
def tree_zeros_like(tree):\n    return jax.tree_util.tree_map(jnp.zeros_like, tree)\n\n\ndef tree_sqnorm(tree):\n    leaves = jax.tree_util.tree_leaves(tree)\n    if not leaves:\n        return jnp.array(0.0)\n    return sum(jnp.sum(jnp.square(x)) for x in leaves)\n\n\ndef tree_l2norm(tree):\n    return jnp.sqrt(tree_sqnorm(tree) + 1e-12)\n\n\ndef clip_tree(tree, max_norm):\n    norm = tree_l2norm(tree)\n    scale = jnp.minimum(1.0, max_norm / (norm + 1e-12))\n    return jax.tree_util.tree_map(lambda x: x * scale, tree)\n\n\ndef adam_init(params):\n    return {"m": tree_zeros_like(params), "v": tree_zeros_like(params),\n            "t": jnp.array(0, dtype=jnp.int32)}\n\n\ndef adam_step(params, state, grads, lr, beta1=0.9, beta2=0.999, eps=1e-8):\n    t = state["t"] + 1\n    m = jax.tree_util.tree_map(lambda m, g: beta1 * m + (1.0 - beta1) * g,\n                                state["m"], grads)\n    v = jax.tree_util.tree_map(lambda v, g: beta2 * v + (1.0 - beta2) * jnp.square(g),\n                                state["v"], grads)\n    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1 ** t), m)\n    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2 ** t), v)\n    params = jax.tree_util.tree_map(\n        lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps),\n        params, m_hat, v_hat)\n    return params, {"m": m, "v": v, "t": t}

## Data: 12×12 downsampled MNIST 0/1\n\nThis matches §13.3 of the note exactly. The downsampling step (4×4 average pooling of 48×48 inputs, or equivalent slicing of 28×28) keeps the visible dimension at 144, which is small enough that prototype storage is cheap but still rich enough that the model must learn structure rather than memorise raw pixels.\n\nPixels are mapped to $[-1, 1]$ so that the OU prior $\\mathcal N(0, k_BT I)$ at $t = \\tau$ has roughly the right scale. This is the implicit assumption behind using $k_BT = 1$ throughout — if you change the data normalisation, scale $k_BT$ accordingly.

In [ ]:
def _load_mnist_raw():\n    """Load raw MNIST training data as (n, 28, 28) uint8 and (n,) labels.\n\n    Tries tensorflow.keras.datasets first (works fully offline once cached),\n    then sklearn.fetch_openml as a fallback.\n    """\n    try:\n        from tensorflow.keras.datasets import mnist\n        (x_train, y_train), _ = mnist.load_data()\n        return np.asarray(x_train, dtype=np.uint8), np.asarray(y_train, dtype=np.int32)\n    except Exception as exc_tf:\n        print("keras.datasets.mnist failed:", exc_tf)\n        from sklearn.datasets import fetch_openml\n        d = fetch_openml("mnist_784", version=1, as_frame=False, cache=True)\n        x = np.asarray(d.data, dtype=np.uint8).reshape(-1, 28, 28)\n        y = np.asarray(d.target, dtype=np.int32)\n        return x[:60000], y[:60000]\n\n\ndef downsample_avg(x_28, out_side=IMG_SIDE):\n    """Downsample (n, 28, 28) uint8 -> (n, out_side, out_side) float32 by\n    cropping to 24x24 and then mean-pooling 2x2.\n\n    24/2 = 12, which is the exact target side for DIGITS=0/1 as in the note.\n    For other out_side values this raises so the implicit assumption is loud.\n    """\n    if out_side != 12:\n        raise ValueError(f"downsample currently hardcoded to 12x12; got {out_side}")\n    x = x_28[:, 2:26, 2:26].astype(np.float32) / 255.0\n    x = x.reshape(-1, 12, 2, 12, 2).mean(axis=(2, 4))\n    return x\n\n\ndef load_mnist_subset(digits, n_train, seed=0):\n    x_raw, y_raw = _load_mnist_raw()\n    mask = np.isin(y_raw, np.asarray(digits))\n    x = x_raw[mask]\n    y = y_raw[mask]\n    rng = np.random.default_rng(seed)\n    perm = rng.permutation(x.shape[0])[:n_train]\n    x = downsample_avg(x[perm])\n    y = y[perm]\n    x = 2.0 * x - 1.0\n    x_flat = x.reshape(x.shape[0], -1)\n    return jnp.asarray(x_flat), jnp.asarray(y)\n\n\ntrain_data, train_labels = load_mnist_subset(DIGITS, N_TRAIN, seed=SEED)\nprint("train_data:", train_data.shape,\n      "| mean/std =", float(jnp.mean(train_data)), "/", float(jnp.std(train_data)),\n      "| per-class counts:", {int(d): int(jnp.sum(train_labels == d)) for d in DIGITS})\n\n# Sanity-check: show a small grid of training images.\nfig, axes = plt.subplots(2, 8, figsize=(8.0, 2.2))\nfor i, ax in enumerate(axes.ravel()):\n    img = np.asarray(train_data[i]).reshape(IMG_SIDE, IMG_SIDE)\n    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)\n    ax.set_title(int(train_labels[i]), fontsize=8)\n    ax.axis("off")\nplt.suptitle("12x12 MNIST (digits {}), normalised to [-1, 1]".format(list(DIGITS)))\nplt.tight_layout()\nplt.show()

## OU corruption and denoising target\n\nIdentical to the 2D notebook, generalised to arbitrary visible dimension. The denoising target is\n$$f^\\star(x_t, x_0, t) \\;=\\; -\\frac{k_BT}{\\sigma_t^2}(x_t - a_t x_0),$$\nwith $a_t = e^{-t}$ and $\\sigma_t^2 = k_BT(1 - a_t^2)$.

In [ ]:
def ou_coeffs(t, kBT=1.0):\n    """Return (a_t, sigma_t^2) for x_t = a_t x_0 + sigma_t * eps."""\n    a = jnp.exp(-t)\n    sigma2 = kBT * (1.0 - a * a)\n    sigma2 = jnp.maximum(sigma2, 1e-6)\n    return a, sigma2\n\n\ndef corrupt_ou(key, x0, t, kBT=1.0):\n    """Forward noising. Returns (xt, denoising_force_target). x0 has shape (B, d)."""\n    eps = random.normal(key, x0.shape)\n    a, sigma2 = ou_coeffs(t, kBT)\n    a_b = a if jnp.asarray(a).ndim == 0 else a[:, None]\n    s2_b = sigma2 if jnp.asarray(sigma2).ndim == 0 else sigma2[:, None]\n    xt = a_b * x0 + jnp.sqrt(s2_b) * eps\n    target_force = -kBT * (xt - a_b * x0) / s2_b\n    return xt, target_force

## Categorical DenseAM model\n\nGiven $K$ trainable prototypes $c_a \\in \\mathbb R^d$ and biases $b_a$, the joint energy at noise level $t$ takes the *drift-aware* form\n$$E_\\theta(x, a, t) \\;=\\; \\frac{1}{2\\sigma_t^2}\\lVert x - a_t c_a\\rVert^2 - b_a.$$\nThis matches eq. (5) of the note: the OU-smoothed empirical mixture has Gaussians centred at the *rescaled* memories $a_t \\xi_\\mu$, so the DenseAM prototypes that play the role of $\\xi_\\mu$ must enter the energy through $a_t c_a$, not $c_a$ directly. The categorical-hidden marginal is\n$$\\log p_\\theta(x, t) \\;=\\; -\\frac{1}{2\\sigma_t^2}\\lVert x\\rVert^2 + \\log\\sum_a \\exp\\!\\left[\\frac{a_t\\,x^\\top c_a}{\\sigma_t^2} - \\frac{a_t^2\\lVert c_a\\rVert^2}{2\\sigma_t^2} + b_a\\right] + \\text{const}.$$\n\nThe responsibilities and analytic force follow directly:\n$$r_a(x, t) \\;=\\; \\mathrm{softmax}_a\\!\\left[\\frac{a_t\\,x^\\top c_a}{\\sigma_t^2} - \\frac{a_t^2\\lVert c_a\\rVert^2}{2\\sigma_t^2} + b_a\\right], \\qquad f_\\theta(x, t) \\;=\\; \\frac{1}{\\sigma_t^2}\\sum_a r_a\\,(a_t c_a - x).$$\n\nAs in 2D, we provide both an autodiff and an analytic force implementation, and check that they agree.

In [ ]:
def init_dam(key, data, K=K_DEFAULT, jitter=0.05):\n    """Initialise prototypes at random data points, biases at zero."""\n    kidx, kjit = random.split(key)\n    idx = random.randint(kidx, (K,), minval=0, maxval=data.shape[0])\n    c0 = data[idx] + jitter * random.normal(kjit, (K, data.shape[1]))\n    return {"c": c0, "b": jnp.zeros((K,))}\n\n\ndef dam_log_p(params, x, a_t, sigma2):\n    """log p_theta(x, t) up to const. x is a single sample of shape (d,)."""\n    c = params["c"]\n    quad_x = -0.5 * jnp.sum(x ** 2) / sigma2\n    logits = (a_t * (x @ c.T) - 0.5 * (a_t ** 2) * jnp.sum(c ** 2, axis=-1)) / sigma2 + params["b"]\n    return quad_x + logsumexp(logits)\n\n\ndef dam_responsibilities(params, x_batch, a_t, sigma2):\n    """r_a(x, t) for a batch. Returns (B, K)."""\n    c = params["c"]\n    logits = (a_t * (x_batch @ c.T) - 0.5 * (a_t ** 2) * jnp.sum(c ** 2, axis=-1)[None, :]) / sigma2 \\\n             + params["b"][None, :]\n    return jax.nn.softmax(logits, axis=-1)\n\n\ndef dam_force_analytic(params, x_batch, a_t, sigma2):\n    """f_theta(x, t) = (1/sigma^2) sum_a r_a (a_t c_a - x)."""\n    r = dam_responsibilities(params, x_batch, a_t, sigma2)\n    return (a_t * (r @ params["c"]) - x_batch) / sigma2\n\n\n_dam_grad_single = jax.grad(dam_log_p, argnums=1)\n\n\ndef dam_force_autodiff(params, x_batch, a_t, sigma2):\n    return jax.vmap(lambda x: _dam_grad_single(params, x, a_t, sigma2))(x_batch)\n\n\ndef dam_free_energy(params, x_batch, a_t, sigma2):\n    """F = -log p up to const, for batched x."""\n    return -jax.vmap(lambda x: dam_log_p(params, x, a_t, sigma2))(x_batch)

## Numerical gradient check\n\nThe analytic force expression must agree with `jax.grad` of `log_p` to floating-point precision; otherwise the energy and the force we sample with are silently different and the reverse sampler integrates the wrong vector field.

In [ ]:
key, k_check_init, k_check_x = random.split(key, 3)\np_check = init_dam(k_check_init, train_data, K=K_DEFAULT)\nx_check = random.normal(k_check_x, (32, IMG_DIM))\n\nfor t_test in [0.05, 0.5, 1.5, TAU]:\n    a_t, s2 = ou_coeffs(jnp.asarray(t_test))\n    a_t, s2 = float(a_t), float(s2)\n    f_an = dam_force_analytic(p_check, x_check, a_t, s2)\n    f_ad = dam_force_autodiff(p_check, x_check, a_t, s2)\n    err = float(jnp.max(jnp.abs(f_an - f_ad)))\n    print(f"t={t_test:5.3f} | a_t={a_t:.3f} | sigma_t^2={s2:.3f} | "\n          f"max|autodiff - analytic|={err:.2e}")

## Denoising force-matching training\n\nPer-slice training, from $t = \\tau$ (high noise, nearly Gaussian) down to $t = T_{\\min}$ (low noise, modes resolved), with parameters warm-started from the previous slice. This is the smooth-driving protocol of §8 of the note.\n\nThe whole inner loop is wrapped in `jax.jit`/`jax.lax.scan` for speed. With $d = 144$, the dominant cost per step is the $B \\times K$ logits matrix plus an outer-product backward pass — `BATCH_SIZE * K * IMG_DIM` flops dominate. For `BATCH_SIZE=128, K=64, IMG_DIM=144` that is ~1.2M flops/step, so the whole schedule fits comfortably in a minute on a modern GPU.

In [ ]:
def dam_loss(params, key, data, t, batch_size, reg_scale, kBT_val):\n    k_idx, k_noise = random.split(key)\n    idx = random.randint(k_idx, (batch_size,), minval=0, maxval=data.shape[0])\n    x0 = data[idx]\n    xt, target = corrupt_ou(k_noise, x0, t, kBT_val)\n    a_t, sigma2 = ou_coeffs(t, kBT_val)\n    pred = dam_force_analytic(params, xt, a_t, sigma2)\n    mse = 0.5 * jnp.mean(jnp.sum((pred - target) ** 2, axis=-1))\n    return mse + reg_scale * tree_sqnorm(params)\n\n\n@partial(jax.jit, static_argnames=("batch_size", "steps", "lr"))\ndef _train_slice_scan(params, opt_state, key, data, t, batch_size, steps, lr):\n    """JIT-compiled inner training loop for one diffusion time slice."""\n    def body(carry, _):\n        params, opt_state, key = carry\n        key, sub = random.split(key)\n        loss_val, grads = jax.value_and_grad(dam_loss)(\n            params, sub, data, t, batch_size, REG_SCALE, kBT)\n        grads = clip_tree(grads, GRAD_CLIP)\n        params, opt_state = adam_step(params, opt_state, grads, lr=lr)\n        return (params, opt_state, key), loss_val\n    (params, opt_state, key), losses = jax.lax.scan(\n        body, (params, opt_state, key), xs=None, length=steps)\n    return params, opt_state, key, losses\n\n\ndef train_dam_schedule(key, data, K, t_grid,\n                       steps_per_time=STEPS_PER_TIME,\n                       batch_size=BATCH_SIZE, lr=LR_CAT, verbose=True):\n    key, k_init = random.split(key)\n    params = init_dam(k_init, data, K=K)\n    params_by_idx = [None] * len(t_grid)\n    losses_by_idx = [None] * len(t_grid)\n\n    t0_wall = time.time()\n    for idx in range(len(t_grid) - 1, -1, -1):\n        opt_state = adam_init(params)\n        t = t_grid[idx]\n        params, opt_state, key, losses = _train_slice_scan(\n            params, opt_state, key, data, t, batch_size, steps_per_time, lr)\n        params_by_idx[idx] = params\n        losses_by_idx[idx] = np.asarray(jax.device_get(losses))\n        if verbose:\n            a_t, s2 = ou_coeffs(t)\n            print(f"DAM (K={K}) | t={float(t):.3f} | sigma_t^2={float(s2):.4f} "\n                  f"| loss={float(losses[-1]):.4e}")\n    if verbose:\n        print(f"DAM (K={K}) finished in {time.time() - t0_wall:.1f}s")\n    return params_by_idx, losses_by_idx, key

## Train the default-size DAM\n\nThis is the headline run with `K = K_DEFAULT`. The `K`-sweep cell below uses the same `train_dam_schedule` function.

In [ ]:
key, sub = random.split(key)\nparams_seq, loss_history, key = train_dam_schedule(sub, train_data, K_DEFAULT, TIME_GRID)

## Convergence curves\n\nOne line per time slice, coloured from dark (large $t$, easy) to light (small $t$, hard). With the drift-aware logits in place the small-$t$ slices should still flatten cleanly; if they keep descending after `STEPS_PER_TIME` the model is under-trained or under-capacitated.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.0, 4.0))\ncmap = plt.cm.viridis\nn_slices = len(loss_history)\nfor idx, losses in enumerate(loss_history):\n    if losses is None:\n        continue\n    color = cmap(idx / max(1, n_slices - 1))\n    ax.plot(losses, color=color, alpha=0.85, linewidth=1.0)\nax.set_xlabel("step within slice")\nax.set_ylabel("denoising force-matching loss")\nax.set_yscale("log")\nax.set_title(f"DAM convergence per time slice (K={K_DEFAULT})")\nax.grid(alpha=0.3, which="both")\nsm = plt.cm.ScalarMappable(cmap=cmap,\n                           norm=plt.Normalize(vmin=float(TIME_GRID[0]),\n                                              vmax=float(TIME_GRID[-1])))\nsm.set_array([])\ncbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)\ncbar.set_label("t")\nplt.tight_layout()\nplt.show()

## Reverse-time sampling\n\nIntegrate the reverse OU SDE\n$$dx \\;=\\; [\\,x + 2 f_\\theta(x, s)\\,]\\,dr \\;+\\; \\sqrt{2 k_BT}\\,dw, \\qquad s = \\tau - r,$$\nfrom $x \\sim \\mathcal N(0, k_BT I)$ at $r = 0$ down to $s = T_{\\min}$ at $r = \\tau$. At each step we look up the parameter set for the time slice closest to the current $s$.\n\nThe whole rollout is wrapped in `jax.lax.scan` over a pre-baked stack of per-slice parameters so it stays on-device.